# American Option Pricing with Greeks

Runs `derivatives.american_option_lsm` -- an American option priced via Longstaff-Schwartz Monte Carlo (LSM), with bump-and-reprice Greeks opted in via `greeks=True` (CLAUDE.md section 2's own note on this: delta/gamma/vega/theta/rho, task #15 Phase 4).

Requires a free-tier API key: https://www.pyvar.com#get-api-key

In [ ]:
%load_ext pyvar_jupyter
%pyvar_key eyJ...  # replace with your own key, or set PYVAR_API_KEY before starting the kernel

## A standard reference case

S=K=100, r=5%, σ=20%, τ=1 year is the textbook Longstaff-Schwartz benchmark case -- the American put's own docstring cites a ~6.024 European-price reference at this exact input set, so it's a good sanity check for the American price this call returns (which must be >= that European price -- early-exercise optionality is never worth less).

In [ ]:
%pyvar derivatives.american_option_lsm spot=100.0 strike=100.0 rate=0.05 sigma=0.20 tau=1.0 \
    option_type=put greeks=True

No `n_simulations`/`n_steps`/`seed` given, so this uses the API's own defaults (100,000 paths, 50 steps) -- fine for exploration; pin `seed=` explicitly if you need a reproducible number across runs.

## Reading the Greeks

For a put, `delta` should come back negative (the option loses value as the underlying rises) and `gamma` positive (delta itself becomes less negative as spot rises, at a decreasing rate near the money). Capture the result and check both directly:

In [ ]:
result = %pyvar derivatives.american_option_lsm spot=100.0 strike=100.0 rate=0.05 sigma=0.20 tau=1.0 \
    option_type=put greeks=True seed=41

assert result["delta"] < 0, "a put's delta should be negative"
assert result["gamma"] > 0, "gamma should be positive near the money"
print(f"price={result['price']:.4f}  delta={result['delta']:.4f}  gamma={result['gamma']:.5f}")

## Comparing put vs. call delta

A quick two-call comparison, using the display helper directly (no magics) since this loops over both option types:

In [ ]:
import os

from pyvar_client import Client

client = Client(api_key=os.environ["PYVAR_API_KEY"])

for option_type in ("put", "call"):
    r = client.derivatives.american_option_lsm(
        spot=100.0, strike=100.0, rate=0.05, sigma=0.20, tau=1.0,
        option_type=option_type, greeks=True, seed=41,
    )
    print(f"{option_type:>4}: price={r['price']:.4f}  delta={r['delta']:.4f}")

Expect the call's delta to be positive and the put's negative -- both roughly symmetric in magnitude around at-the-money, per standard option-Greeks behavior (this isn't a pyvar-specific property, just confirming the API surfaces it correctly).